# M3 · Tool de Normalización Terminológica (SNOMED CT vía BioPortal)

**Tópicos Especiales en IA · Universidad EAFIT**  
**Responsable:** Luis  

Este notebook prueba la herramienta de normalización terminológica cumpliendo con el **Nivel 4 (5 puntos)** de la rúbrica:
1. **Integración de herramienta externa** (SNOMED CT vía BioPortal).
2. **Criterio explícito de invocación** (se llama sólo si el score del retrieval de la entidad cruda es insuficiente).
3. **Manejo explícito de fallos** (campo `normalization_failed: True`, nunca fallback silencioso).


## 1 · Configuración y Acceso a Drive

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

RAIZ = Path('/content/drive/MyDrive/TopicosIA/Proyecto-Salud')
if not RAIZ.exists():
    RAIZ = Path('/content/Topicos-IA')

print('Directorio de trabajo:', RAIZ)
%cd {RAIZ}

## 2 · Instalación de Dependencias

In [ ]:
%pip install -q requests python-dotenv PyYAML
print('Dependencias de Tools instaladas con éxito.')

## 3 · Ejecución de la Tool de Normalización
Se evalúan las consultas de prueba. Si no se cuenta con `BIOPORTAL_API_KEY`, corre automáticamente en `MOCK_MODE` sin fallar.

In [ ]:
%cd {RAIZ}/M3/tools
!python run_tools.py --config config.yaml

## 4 · Demostración del Criterio de Invocación y Manejo de Fallos
Verificamos la regla de orquestación ante score alto, score bajo con acierto, y fallo en la ontología.

In [ ]:
import sys
sys.path.insert(0, str(RAIZ / 'M3/tools'))
from orquestacion import resolver_query, _retrieve_mock_bueno, _retrieve_mock_malo
from tool_normalizacion import normalizar_entidad

print('--- CASO 1: Score de retrieval alto (>= 0.5) -> Tool NO se invoca ---')
r1 = resolver_query('diabetes tipo 2', _retrieve_mock_bueno, umbral=0.5)
print(r1)

print('\n--- CASO 2: Score de retrieval bajo (< 0.5) -> Tool SÍ se invoca y normaliza ---')
r2 = resolver_query('diabetes tipo 2', _retrieve_mock_malo, umbral=0.5)
print(r2)

print('\n--- CASO 3: Falla de normalización -> Fallo explícito (normalization_failed: True) ---')
r3 = normalizar_entidad('enfermedad_inexistente_xyz_123')
print(r3)
